# Signisa — tensor prep (CPU, internet ON)

Builds the ragged native-length (T, N, 4) float16 training shards (shard schema v2).
Attach the Kaggle **asl-signs** competition data, and — for a dual-domain build — each
shard's LATEST `kaggle_extract.ipynb` output (chained versions carry the shard's complete
corpus, so one input per shard; discovered by globbing for `shard_manifest.json`.
Attaching two versions of the same shard trips the overlapping-stems assert on purpose). ASL Citizen clips are gloss-mapped to the 246 canonical classes,
signers namespaced `ac_*`, and merged with a `domain` column. With no extraction outputs
attached this degrades to the plain PopSign build.
Publish this notebook's output and attach it to `kaggle_train.ipynb`.

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
LANDMARK_VERSION = "v2"  # v2 = 99 nodes with the 40-point lip set (task3a lever)

# discover extraction-notebook outputs among the attached inputs
from pathlib import Path

manifests = sorted(Path("/kaggle/input").glob("*/shard_manifest.json"))
CITIZEN_DIRS = [m.parent / "extracted" for m in manifests]
SPLITS = next((m.parent / "splits" for m in manifests if (m.parent / "splits").is_dir()), None)
import json as _json
for m in manifests:
    print(m.parent.name, _json.load(m.open()))
assert not CITIZEN_DIRS or SPLITS is not None, "extraction outputs lack a splits/ dir"
metas = [_json.load(m.open()) for m in manifests]
seen = sorted({m["shard_index"] for m in metas})
if metas:
    n = metas[0]["num_shards"]
    if seen != list(range(n)):
        print(f"NOTE: shards attached {seen} of {n} — building from a PARTIAL extraction")
    # a chained shard is complete only in its LAST version; older links report remaining>0
    open_chains = {m["shard_index"] for m in metas if m.get("n_remaining", 0) > 0}
    finished = {m["shard_index"] for m in metas if m.get("n_remaining", 1) == 0}
    if open_chains - finished:
        print(f"NOTE: shards {sorted(open_chains - finished)} have clips remaining — "
              "attach their latest chained version or accept a PARTIAL build")

In [ ]:
# gloss-map ASL Citizen clips to the canonical classes (skipped when none attached)
if CITIZEN_DIRS:
    !python /kaggle/working/signisa-repo/scripts/map_asl_citizen.py \
        --csv-dir {SPLITS} \
        --labels /kaggle/working/signisa-repo/data/meta/training_labels.json \
        --curriculum /kaggle/working/signisa-repo/data/meta/curriculum_db.json \
        --out-dir /kaggle/working/meta

In [ ]:
citizen_flags = " ".join(
    [f"--citizen-npz-dir {d}" for d in CITIZEN_DIRS]
    + (["--citizen-mapping /kaggle/working/meta/asl_citizen_mapping.csv"] if CITIZEN_DIRS else []))

!python /kaggle/working/signisa-repo/scripts/build_training_tensors.py \
    --train-csv /kaggle/input/asl-signs/train.csv \
    --landmarks-dir /kaggle/input/asl-signs \
    --labels /kaggle/working/signisa-repo/data/meta/training_labels.json \
    --out-dir /kaggle/working/tensors \
    --landmark-version {LANDMARK_VERSION} {citizen_flags}

In [ ]:
from pathlib import Path
import pandas as pd

tensors = Path("/kaggle/working/tensors")
index = pd.read_csv(tensors / "index.csv", dtype={"participant_id": str})
size_gb = sum(f.stat().st_size for f in tensors.iterdir()) / 1e9
print(f"{len(index)} sequences, {size_gb:.2f} GB")
print(index.groupby("domain").agg(sequences=("sequence_id", "size"),
                                  signers=("participant_id", "nunique")))
pop = set(index[index.domain == "popsign"].participant_id)
cit = set(index[index.domain == "asl_citizen"].participant_id)
assert not pop & cit, f"participant-id collision across domains: {sorted(pop & cit)[:5]}"
assert all(p.startswith("ac_") for p in cit)
assert size_gb < 15, "over the notebook-output budget"